# 1. Read Libraries and Data

In [1]:
import pandas as pd
import numpy as np

import optuna
import tensorflow as tf

""" Sequential -  It allows you to create a model by simply adding layers one after another.
This is used for building simple neural networks where the flow of data is straightforward, 
from input to output, through a series of layers."""
from tensorflow.keras.models import Sequential

"""Dense is a fully connected layer, also known as a dense layer, in a neural network.
Each neuron in a dense layer receives input from all neurons in the previous layer."""
from tensorflow.keras.layers import Dense

"""Optimizers computes adaptive learning rates for each parameter"""
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta, Nadam

from sklearn.metrics import roc_auc_score

data = pd.read_csv('Employee.csv')

data

C:\Users\aksta\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
0,Bachelors,2017,Bangalore,3,34,Male,No,0,0
1,Bachelors,2013,Pune,1,28,Female,No,3,1
2,Bachelors,2014,New Delhi,3,38,Female,No,2,0
3,Masters,2016,Bangalore,3,27,Male,No,5,1
4,Masters,2017,Pune,3,24,Male,Yes,2,1
...,...,...,...,...,...,...,...,...,...
4648,Bachelors,2013,Bangalore,3,26,Female,No,4,0
4649,Masters,2013,Pune,2,37,Male,No,2,1
4650,Masters,2018,New Delhi,3,27,Male,No,5,1
4651,Bachelors,2012,Bangalore,3,30,Male,Yes,2,0


# 2. Descriptive Statistics

In [2]:
data.describe(include='all')

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain,LeaveOrNot
count,4653,4653.000000,4653,4653.000000,4653.000000,4653,4653,4653.000000,4653.000000
unique,3,NaN,3,NaN,NaN,2,2,NaN,NaN
top,Bachelors,NaN,Bangalore,NaN,NaN,Male,No,NaN,NaN
freq,3601,NaN,2228,NaN,NaN,2778,4175,NaN,NaN
mean,NaN,2015.062970,NaN,2.698259,29.393295,NaN,NaN,2.905652,0.343864
std,NaN,1.863377,NaN,0.561435,4.826087,NaN,NaN,1.558240,0.475047
min,NaN,2012.000000,NaN,1.000000,22.000000,NaN,NaN,0.000000,0.000000
25%,NaN,2013.000000,NaN,3.000000,26.000000,NaN,NaN,2.000000,0.000000
50%,NaN,2015.000000,NaN,3.000000,28.000000,NaN,NaN,3.000000,0.000000
75%,NaN,2017.000000,NaN,3.000000,32.000000,NaN,NaN,4.000000,1.000000


# 3. Declare target and inputs and Split data using train_test_split

In [3]:
from sklearn.model_selection import train_test_split

inputs = data.drop('LeaveOrNot', axis=1)
target = data['LeaveOrNot']

X_train, X_test, y_train, y_test = train_test_split(inputs, target, test_size=0.2, random_state=42)

# 4. Using Label encoder convert categorics to numerical

In [4]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
for i in X_train.select_dtypes(include=['object', 'string']).columns:
    le = LabelEncoder()
    X_train[i] = le.fit_transform(X_train[i])
    encoders[i] = le

for i in X_test.columns:
    if i in encoders:
        le = encoders[i]
        X_test[i] = le.transform(X_test[i])

display(X_train)
display(X_test)

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain
2850,1,2013,1,3,30,1,0,0
589,0,2012,0,3,25,1,0,3
2086,1,2017,2,2,29,1,0,2
445,1,2012,2,3,24,1,0,2
3654,1,2017,1,2,35,1,0,2
...,...,...,...,...,...,...,...,...
4426,0,2014,0,3,32,1,0,1
466,0,2013,0,3,26,1,1,4
3092,0,2016,0,3,39,0,0,1
3772,0,2015,0,3,41,1,0,1


,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain
297,0,2016,0,3,24,0,0,2
2705,0,2013,0,3,26,0,0,4
501,0,2017,1,2,25,0,0,3
1272,1,2015,2,2,28,0,0,2
3956,0,2012,0,3,33,1,0,1
...,...,...,...,...,...,...,...,...
3103,1,2017,1,1,40,1,0,5
733,0,2013,0,3,25,1,0,3
1419,0,2018,2,3,26,1,1,4
3291,0,2012,0,3,37,1,0,4


# 5. Scale inputs using StandartScaler

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training data
scaler.fit(X_train)

# Transform train and test
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

display(X_train_scaled)
display(X_test_scaled)

,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain
2850,1.408598,-1.109740,0.252181,0.539424,0.117526,0.815674,-0.333731,-1.861550
589,-0.510436,-1.645675,-0.941990,0.539424,-0.914641,0.815674,-0.333731,0.061305
2086,1.408598,1.034001,1.446353,-1.226396,-0.088907,0.815674,-0.333731,-0.579646
445,1.408598,-1.645675,1.446353,0.539424,-1.121074,0.815674,-0.333731,-0.579646
3654,1.408598,1.034001,0.252181,-1.226396,1.149693,0.815674,-0.333731,-0.579646
...,...,...,...,...,...,...,...,...
4426,-0.510436,-0.573805,-0.941990,0.539424,0.530393,0.815674,-0.333731,-1.220598
466,-0.510436,-1.109740,-0.941990,0.539424,-0.708208,0.815674,2.996423,0.702257
3092,-0.510436,0.498065,-0.941990,0.539424,1.975427,-1.225980,-0.333731,-1.220598
3772,-0.510436,-0.037870,-0.941990,0.539424,2.388294,0.815674,-0.333731,-1.220598


,Education,JoiningYear,City,PaymentTier,Age,Gender,EverBenched,ExperienceInCurrentDomain
297,-0.510436,0.498065,-0.941990,0.539424,-1.121074,-1.225980,-0.333731,-0.579646
2705,-0.510436,-1.109740,-0.941990,0.539424,-0.708208,-1.225980,-0.333731,0.702257
501,-0.510436,1.034001,0.252181,-1.226396,-0.914641,-1.225980,-0.333731,0.061305
1272,1.408598,-0.037870,1.446353,-1.226396,-0.295341,-1.225980,-0.333731,-0.579646
3956,-0.510436,-1.645675,-0.941990,0.539424,0.736827,0.815674,-0.333731,-1.220598
...,...,...,...,...,...,...,...,...
3103,1.408598,1.034001,0.252181,-2.992215,2.181861,0.815674,-0.333731,1.343209
733,-0.510436,-1.109740,-0.941990,0.539424,-0.914641,0.815674,-0.333731,0.061305
1419,-0.510436,1.569936,1.446353,0.539424,-0.708208,0.815674,2.996423,0.702257
3291,-0.510436,-1.645675,-0.941990,0.539424,1.562560,0.815674,-0.333731,0.702257


# 6. Create ANN model for classification use with best hyperparameters using Optuna

In [6]:
def create_model(trial):
    # Building artificial neural network
    model = Sequential()

     # we add 2 hidden layers and 1 output layer
    model.add(Dense(units=trial.suggest_int('units_layer1', 6, 32), activation='relu'))
    model.add(Dense(units=trial.suggest_int('units_layer2', 6, 32), activation='relu'))
    model.add(Dense(units=1, activation='sigmoid'))

    # Suggest hyperparameters for the optimizer
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop', 'adagrad'])
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        optimizer = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    elif optimizer_name == 'adagrad':
        optimizer = Adagrad(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['AUC'])
    
    return model

def optimal(trial):
    
    # Suggest the number of epochs and batch size
    epochs = trial.suggest_int('epochs', 10, 50)
    batch_size = trial.suggest_int('batch_size', 16, 64)
    
    model = create_model(trial)
    
    history = model.fit(X_train_scaled, y_train, epochs=epochs, batch_size=batch_size)
    
    y_pred = model.predict(X_test_scaled)
    score = roc_auc_score(y_test, y_pred)
    
    return score

study = optuna.create_study(direction='maximize')
study.optimize(optimal, n_trials=10)

print(f"Best trial: {study.best_trial.value}")
print(f"Best hyperparameters: {study.best_trial.params}")

best_params = study.best_trial.params

best_params

[I 2026-10-17 10:23:26,217] A new study created in memory with name: no-name-bcca2186-7c40-4bc4-93cc-9f8459457f08
C:\Users\aksta\AppData\Local\Temp\ipykernel_32492\1466992178.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


Epoch 1/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.6930 - loss: 0.5828
Epoch 2/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7705 - loss: 0.5146  
Epoch 3/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7980 - loss: 0.4833
Epoch 4/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8168 - loss: 0.4593  
Epoch 5/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8282 - loss: 0.4422  
Epoch 6/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8362 - loss: 0.4286  
Epoch 7/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8427 - loss: 0.4178  
Epoch 8/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8482 - loss: 0.4085  
Epoch 9/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8526 - loss: 0.4009  
Epoch 10/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8560 - loss: 0.3951  
Epoch 11/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8590 - loss: 0.3906
Epoch 12/29
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8614 - loss: 0.3869
Epoch 13/29
83/83 ━━━━━━━━━━━━━━━

[I 2026-10-17 10:23:31,464] Trial 0 finished with value: 0.8915836780552575 and parameters: {'epochs': 29, 'batch_size': 45, 'units_layer1': 16, 'units_layer2': 31, 'optimizer': 'rmsprop', 'learning_rate': 0.0035811641608295528}. Best is trial 0 with value: 0.8915836780552575.


Epoch 1/44


C:\Users\aksta\AppData\Local\Temp\ipykernel_32492\1466992178.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


...

C:\Users\aksta\AppData\Local\Temp\ipykernel_32492\1466992178.py:12: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.4011 - loss: 0.6967
Epoch 2/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.5056 - loss: 0.6561  
Epoch 3/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.6203 - loss: 0.6281
Epoch 4/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.6725 - loss: 0.6094  
Epoch 5/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.6941 - loss: 0.5956  
Epoch 6/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7079 - loss: 0.5851  
Epoch 7/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7184 - loss: 0.5767  
Epoch 8/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7273 - loss: 0.5695  
Epoch 9/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7344 - loss: 0.5632
Epoch 10/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7408 - loss: 0.5573  
Epoch 11/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7467 - loss: 0.5519
Epoch 12/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7520 - loss: 0.5469  
Epoch 13/31
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 1m

[I 2026-10-17 10:24:39,748] Trial 9 finished with value: 0.8398243194933864 and parameters: {'epochs': 31, 'batch_size': 53, 'units_layer1': 15, 'units_layer2': 9, 'optimizer': 'rmsprop', 'learning_rate': 0.0005136449402696853}. Best is trial 3 with value: 0.8919871303814922.


Best trial: 0.8919871303814922
Best hyperparameters: {'epochs': 48, 'batch_size': 63, 'units_layer1': 29, 'units_layer2': 13, 'optimizer': 'adam', 'learning_rate': 0.0013669388891433209}


{'epochs': 48,
 'batch_size': 63,
 'units_layer1': 29,
 'units_layer2': 13,
 'optimizer': 'adam',
 'learning_rate': 0.0013669388891433209}

# 7. Train and evaluate the final model with the best hyperparameters

In [7]:
# Train the final model with the best hyperparameters

best_model = Sequential()
best_model.add(Dense(units=best_params['units_layer1'], activation='relu'))
best_model.add(Dense(units=best_params['units_layer2'], activation='relu'))
best_model.add(Dense(1, activation='sigmoid'))

if best_params['optimizer'] == 'adam':
    best_optimizer = Adam(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'sgd':
    best_optimizer = SGD(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'rmsprop':
    best_optimizer = RMSprop(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'adagrad':
    best_optimizer = Adagrad(learning_rate=best_params['learning_rate'])

best_model.compile(optimizer=best_optimizer, loss='binary_crossentropy', metrics=['AUC'])


def evaluate(model, X_train, y_train, X_test, y_test):
    
    model.fit(X_train, y_train, epochs=best_params['epochs'], batch_size=best_params['batch_size'])
    
    '''Predictions and probabilities for the training set'''
    
    y_train_prob = model.predict(X_train)

    '''Predictions and probabilities for the test set'''
    
    y_test_prob = model.predict(X_test)

    '''Calculate metrics for the training set''' 
    
    roc_train_prob = roc_auc_score(y_train, y_train_prob)
    gini_train_prob = roc_train_prob * 2 - 1
    

    '''Calculate metrics for the test set'''
    
    roc_test_prob = roc_auc_score(y_test, y_test_prob)
    gini_test_prob = roc_test_prob * 2 - 1
    

    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'Gini': [gini_train_prob * 100, gini_test_prob * 100],
    
    })

    return results


evaluate(best_model, X_train_scaled, y_train, X_test_scaled, y_test)

Epoch 1/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - AUC: 0.6419 - loss: 0.6257
Epoch 2/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7424 - loss: 0.5561  
Epoch 3/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7660 - loss: 0.5311  
Epoch 4/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7833 - loss: 0.5113  
Epoch 5/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.7974 - loss: 0.4951  
Epoch 6/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8105 - loss: 0.4807  
Epoch 7/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8204 - loss: 0.4690  
Epoch 8/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8264 - loss: 0.4596
Epoch 9/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8317 - loss: 0.4515  
Epoch 10/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8362 - loss: 0.4438  
Epoch 11/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8402 - loss: 0.4370  
Epoch 12/48
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8438 - loss: 0.4308
Epoch 13/48
60/60 ━━━━━━━━━━━━━

,Dataset,Gini
0,Train,77.823828
1,Test,77.351514
